In [2]:
import asyncio
import json
import logging
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import requests
from apscheduler.schedulers.blocking import BlockingScheduler
from playwright.sync_api import TimeoutError as PlaywrightTimeoutError
from playwright.sync_api import sync_playwright

# 執行模式：'manual' 立即執行一次，'schedule' 啟動常駐排程
RUN_MODE = 'manual'
USE_EXISTING_CHROME = True
CHROME_CDP_URL = 'http://127.0.0.1:9222'

CONFIG_PATH = Path('config.json')
LOG_DIR = Path('logs')
CAPTCHA_DIR = Path('runtime') / 'captcha'
scheduler = None

@dataclass
class SelectorConfig:
    username: str
    password: str
    captcha_input: str
    login_submit: str
    target_button: str
    captcha_image: str | None = None
    post_login_ready: str | None = None
    target_done: str | None = None

@dataclass
class SiteConfig:
    login_url: str
    username: str
    password: str
    browser: str
    headless: bool

@dataclass
class ScheduleConfig:
    times: list[str]
    days: list[str]
    timezone: str

@dataclass
class TimingConfig:
    navigation_timeout_ms: int
    wait_timeout_ms: int

@dataclass
class RetryConfig:
    attempts: int
    wait_seconds: int

@dataclass
class CaptchaConfig:
    input_mode: str
    wait_seconds: int
    poll_seconds: int

@dataclass
class LineConfig:
    enabled: bool
    channel_access_token: str
    user_id: str
    public_base_url: str
    inbox_file: str

@dataclass
class AppConfig:
    schedule: ScheduleConfig
    site: SiteConfig
    selectors: SelectorConfig
    timing: TimingConfig
    retry: RetryConfig
    captcha: CaptchaConfig
    line: LineConfig

DAY_MAP = {'mon': 'mon', 'tue': 'tue', 'wed': 'wed', 'thu': 'thu', 'fri': 'fri', 'sat': 'sat', 'sun': 'sun'}

def setup_logging() -> None:
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    CAPTCHA_DIR.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', handlers=[logging.FileHandler(LOG_DIR / f'run-{datetime.now():%Y%m%d}.log', encoding='utf-8'), logging.StreamHandler(sys.stdout)], force=True)

def load_config(path: Path) -> AppConfig:
    if not path.exists():
        raise FileNotFoundError('config.json not found')
    raw: dict[str, Any] = json.loads(path.read_text(encoding='utf-8'))
    return AppConfig(ScheduleConfig(**raw['schedule']), SiteConfig(**raw['site']), SelectorConfig(**raw['selectors']), TimingConfig(**raw['timing']), RetryConfig(**raw['retry']), CaptchaConfig(**raw.get('captcha', {})), LineConfig(**raw.get('line', {})))

def validate_schedule(config: ScheduleConfig) -> None:
    if not config.times or not config.days:
        raise ValueError('schedule.times and schedule.days cannot be empty')
    for day in config.days:
        if day not in DAY_MAP:
            raise ValueError(f'Unsupported day value: {day}')
    for value in config.times:
        parts = value.split(':')
        if len(parts) != 2 or not all(part.isdigit() for part in parts):
            raise ValueError(f'Invalid time format: {value}')
        hour, minute = map(int, parts)
        if not (0 <= hour <= 23 and 0 <= minute <= 59):
            raise ValueError(f'Invalid time value: {value}')

def get_chrome_profile() -> tuple[Path, str] | None:
    candidates = [
        Path.home() / 'AppData/Local/Google/Chrome/User Data',
        Path.home() / 'AppData/Local/Google/Chrome SxS/User Data',
        Path.home() / 'AppData/Local/Microsoft/Edge/User Data',
    ]
    for user_data_dir in candidates:
        if not user_data_dir.exists():
            continue
        profile_name = 'Default'
        state_path = user_data_dir / 'Local State'
        try:
            state = json.loads(state_path.read_text(encoding='utf-8'))
            profile_name = state.get('profile', {}).get('last_used', 'Default')
        except (OSError, json.JSONDecodeError):
            pass
        if (user_data_dir / profile_name).exists():
            return user_data_dir, profile_name
        if (user_data_dir / 'Default').exists():
            return user_data_dir, 'Default'
    return None

def is_placeholder_value(value: str | None) -> bool:
    if value is None:
        return True
    cleaned = value.strip().lower()
    return not cleaned or cleaned in {'your_account', 'yourpassword', 'your_username', 'your_user', 'username', 'password'} or cleaned.startswith('your_')

def load_line_replies(inbox_path: Path) -> list[dict[str, Any]]:
    if not inbox_path.exists():
        return []
    entries = []
    for raw_line in inbox_path.read_text(encoding='utf-8').splitlines():
        try:
            if raw_line.strip():
                entries.append(json.loads(raw_line))
        except json.JSONDecodeError:
            continue
    return entries

def build_captcha_image(page, config: AppConfig) -> Path:
    file_path = CAPTCHA_DIR / f'captcha-{datetime.now():%Y%m%d-%H%M%S}.png'
    if config.selectors.captcha_image:
        page.locator(config.selectors.captcha_image).first.screenshot(path=str(file_path))
    else:
        page.screenshot(path=str(file_path), full_page=True)
    latest_path = CAPTCHA_DIR / 'current.png'
    latest_path.write_bytes(file_path.read_bytes())
    logging.info('Saved captcha image: %s', latest_path)
    return latest_path

def send_line_captcha(config: AppConfig, image_url: str) -> None:
    response = requests.post('https://api.line.me/v2/bot/message/push', headers={'Authorization': f'Bearer {config.line.channel_access_token}', 'Content-Type': 'application/json'}, json={'to': config.line.user_id, 'messages': [{'type': 'text', 'text': '請回覆此驗證碼訊息的數字，我會自動填入網頁。'}, {'type': 'image', 'originalContentUrl': image_url, 'previewImageUrl': image_url}]}, timeout=15)
    if response.status_code >= 300:
        raise RuntimeError(f'LINE push failed: {response.status_code} {response.text}')

def wait_for_line_captcha(config: AppConfig, request_time: datetime) -> str:
    deadline = time.time() + config.captcha.wait_seconds
    inbox_path = Path(config.line.inbox_file)
    while time.time() < deadline:
        for item in reversed(load_line_replies(inbox_path)):
            try:
                created_at = datetime.fromisoformat(item.get('created_at', ''))
            except ValueError:
                continue
            text = str(item.get('text', '')).strip()
            if text and created_at >= request_time:
                return ''.join(re.findall(r'\d+', text)) or text
        time.sleep(max(config.captcha.poll_seconds, 1))
    raise TimeoutError('Timed out waiting for LINE captcha reply')

def get_captcha_text(page, config: AppConfig) -> str:
    mode = config.captcha.input_mode.lower().strip()
    if mode == 'manual':
        text = input('請輸入瀏覽器中的 CAPTCHA：').strip()
        if not text:
            raise ValueError('Captcha cannot be empty')
        return text
    if mode == 'line':
        if not config.line.enabled or not config.line.channel_access_token or not config.line.user_id or not config.line.public_base_url:
            raise ValueError('LINE settings are incomplete')
        build_captcha_image(page, config)
        request_time = datetime.now()
        image_url = f"{config.line.public_base_url.rstrip('/')}/captcha/current.png?t={int(request_time.timestamp())}"
        send_line_captcha(config, image_url)
        return wait_for_line_captcha(config, request_time)
    raise ValueError(f'Unsupported captcha input mode: {config.captcha.input_mode}')

def take_error_screenshot(page, prefix: str) -> None:
    file_path = LOG_DIR / f'{prefix}-{datetime.now():%Y%m%d-%H%M%S}.png'
    try:
        page.screenshot(path=str(file_path), full_page=True)
        logging.info('Saved screenshot: %s', file_path)
    except Exception:
        logging.warning('Could not save error screenshot')

def run_once(config: AppConfig) -> None:
    logging.info('Job started')
    with sync_playwright() as playwright:
        browser_type = getattr(playwright, config.site.browser, None)
        if browser_type is None:
            raise ValueError(f'Unsupported browser type: {config.site.browser}')
        browser = None
        context = None
        page = None
        attached_to_existing = False
        try:
            if USE_EXISTING_CHROME and config.site.browser.lower() in {'chromium', 'chrome'}:
                try:
                    browser = browser_type.connect_over_cdp(CHROME_CDP_URL)
                    context = browser.contexts[0] if browser.contexts else browser.new_context()
                    attached_to_existing = True
                    logging.info('Connected to existing Chrome at %s', CHROME_CDP_URL)
                except Exception:
                    browser = None
                    context = None
            if context is None:
                profile = get_chrome_profile()
                if profile and config.site.browser.lower() in {'chromium', 'chrome'}:
                    user_data_dir, profile_name = profile
                    logging.info('Using Chrome profile: %s (%s)', user_data_dir, profile_name)
                    try:
                        context = browser_type.launch_persistent_context(str(user_data_dir), headless=config.site.headless, channel='chrome', args=[f'--profile-directory={profile_name}'])
                    except Exception as error:
                        raise RuntimeError('Chrome 個人資料目前被使用中，請先完全關閉所有 Chrome 視窗，或用 remote debugging 模式啟動 Chrome。') from error
                else:
                    browser = browser_type.launch(headless=config.site.headless)
                    context = browser.new_context()
            page = context.new_page()
            page.goto(config.site.login_url, wait_until='domcontentloaded', timeout=config.timing.navigation_timeout_ms)
            username = page.locator(config.selectors.username)
            password = page.locator(config.selectors.password)
            if is_placeholder_value(config.site.username) or is_placeholder_value(config.site.password):
                username.click()
                username.focus()
                if username.input_value() == '':
                    username.press('Tab')
                password.click()
                password.focus()
                if password.input_value() == '':
                    page.keyboard.press('Tab')
            else:
                username.fill(config.site.username)
                password.fill(config.site.password)
            if config.selectors.captcha_input:
                page.fill(config.selectors.captcha_input, get_captcha_text(page, config))
            page.click(config.selectors.login_submit)
            if config.selectors.post_login_ready:
                page.wait_for_selector(config.selectors.post_login_ready, timeout=config.timing.wait_timeout_ms)
            page.click(config.selectors.target_button)
            if config.selectors.target_done:
                page.wait_for_selector(config.selectors.target_done, timeout=config.timing.wait_timeout_ms)
            logging.info('Target button clicked successfully')
        except Exception:
            if page is not None:
                take_error_screenshot(page, 'error')
            raise
        finally:
            if context is not None and not attached_to_existing:
                context.close()
            if browser is not None and not attached_to_existing:
                browser.close()

def run_with_retry(config: AppConfig) -> None:
    last_error = None
    for attempt in range(1, config.retry.attempts + 1):
        try:
            logging.info('Attempt %s/%s', attempt, config.retry.attempts)
            run_once(config)
            return
        except (PlaywrightTimeoutError, ValueError, RuntimeError) as error:
            last_error = error
            logging.error('Attempt failed: %s', error)
            if attempt < config.retry.attempts:
                time.sleep(config.retry.wait_seconds)
    if last_error:
        raise last_error

def run_in_notebook(config: AppConfig) -> None:
    previous_policy = asyncio.get_event_loop_policy()
    try:
        if sys.platform == 'win32':
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        with ThreadPoolExecutor(max_workers=1) as executor:
            executor.submit(run_with_retry, config).result()
    finally:
        asyncio.set_event_loop_policy(previous_policy)

def start_schedule(config: AppConfig) -> None:
    global scheduler
    scheduler = BlockingScheduler(timezone=ZoneInfo(config.schedule.timezone))
    day_expr = ','.join(DAY_MAP[day] for day in config.schedule.days)
    for hhmm in config.schedule.times:
        hour, minute = hhmm.split(':')
        scheduler.add_job(run_with_retry, 'cron', day_of_week=day_expr, hour=int(hour), minute=int(minute), args=[config], id=f'job-{hhmm}', replace_existing=True)
        print(f'已排程：星期 {day_expr} {hhmm}')
    scheduler.start()

def stop_schedule() -> None:
    if scheduler is not None and scheduler.running:
        scheduler.shutdown(wait=False)
        print('排程已停止')
    else:
        print('目前沒有正在執行的排程')

setup_logging()
config = load_config(CONFIG_PATH)
validate_schedule(config.schedule)
print(f'目前網站：{config.site.login_url}')
print(f'排程：星期 {config.schedule.days}，時間 {config.schedule.times}（{config.schedule.timezone}）')

if RUN_MODE == 'manual':
    run_in_notebook(config)
elif RUN_MODE == 'schedule':
    start_schedule(config)
else:
    raise ValueError("RUN_MODE 必須是 'manual' 或 'schedule'")

目前網站：https://pk12.cloudhr.tw/login.aspx
排程：星期 ['mon', 'tue', 'wed', 'thu', 'fri']，時間 ['07:30']（Asia/Taipei）
2026-09-18 08:47:03,414 | INFO | Attempt 1/2
2026-09-18 08:47:03,417 | INFO | Job started


C:\Users\何裕隆\AppData\Local\Temp\ipykernel_8964\454196865.py:294: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  previous_policy = asyncio.get_event_loop_policy()
C:\Users\何裕隆\AppData\Local\Temp\ipykernel_8964\454196865.py:297: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\何裕隆\AppData\Local\Temp\ipykernel_8964\454196865.py:297: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


2026-09-18 08:47:04,098 | INFO | Using Chrome profile: C:\Users\何裕隆\AppData\Local\Google\Chrome\User Data (Default)
2026-09-18 08:47:04,320 | ERROR | Attempt failed: Chrome 個人資料目前被使用中，請先完全關閉所有 Chrome 視窗，或用 remote debugging 模式啟動 Chrome。
2026-09-18 08:47:09,322 | INFO | Attempt 2/2
2026-09-18 08:47:09,323 | INFO | Job started
2026-09-18 08:47:09,950 | INFO | Using Chrome profile: C:\Users\何裕隆\AppData\Local\Google\Chrome\User Data (Default)
2026-09-18 08:47:10,191 | ERROR | Attempt failed: Chrome 個人資料目前被使用中，請先完全關閉所有 Chrome 視窗，或用 remote debugging 模式啟動 Chrome。


C:\Users\何裕隆\AppData\Local\Temp\ipykernel_8964\454196865.py:301: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(previous_policy)


RuntimeError: Chrome 個人資料目前被使用中，請先完全關閉所有 Chrome 視窗，或用 remote debugging 模式啟動 Chrome。